# Dev26 - NDI Analysis Using Utils Module

This notebook demonstrates the same NDI analysis as dev25, but using the refactored `ndi_analysis_utils.py` module.

**Workflow:**
1. Load transect data
2. Apply preprocessing (illumination correction, wavelength filtering, smoothing, normalization)
3. Compute NDI maps using utils functions
4. Visualize results

**Date:** November 3, 2025

In [ ]:
# Import required libraries
import importlib
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Add parent directory to path
sys.path.append(os.path.abspath("../"))

# Import gref4hsi modules
from utils.gref_pipeline import georef
from gref_pipeline import config

importlib.reload(georef)
from utils.gref_pipeline.georef import *

# Import NDI analysis utilities
from utils.ndi_analysis_utils import *

print("✅ All modules loaded successfully!")

## 1. Load Transect Data

In [ ]:
# Load 028 transect
transect = load_transect(r"E:\mjosa_new_oct_2025\use_gref4hsi\028\output")
transect.list_files()

# Select 028_5 for NDI analysis
cube = transect.select_files(
    [
        "rad_uhi_20241029_125028_5",
    ]
)
cube.describe()

## 2. Preprocessing Pipeline

### 2.1 Apply Illumination Correction

In [ ]:
# Apply illumination correction
cube.apply_illumination_correction_v2()

print(f"✅ Illumination correction complete")
print(f"   Data shape: {cube.data_corrected.shape}")

### 2.2 Visualize Transect (RGB)

In [ ]:
cube.plot_rgb()

In [ ]:
cube.plot_rgb(use_corrected=True)

In [ ]:
# Plot RGB to visualize the transect
# %matplotlib qt

cube.plot_georef(
    apply_alignment_shift=True,
    use_corrected=True,
    coordinate_system="NED",
    figsize=(40, 10),
)

print(f"\n📊 Transect 028_5 overview")
print(f"   Tracks: {cube.data_corrected.shape[0]}")
print(f"   Slits: {cube.data_corrected.shape[1]}")
print(f"   Wavelengths: {cube.data_corrected.shape[2]}")

### 2.3 Crop Wavelengths (490-690 nm)

In [ ]:
# Step 1: Crop wavelengths to 490-690 nm
cube.apply_wavelength_filter(wavelength_range=(490, 690))

print(f"✅ Wavelength cropping complete")
print(f"   Cropped shape: {cube.data_corrected.shape}")
print(f"   Wavelength range: {cube.wavelengths[0]:.1f} - {cube.wavelengths[-1]:.1f} nm")
print(f"   Number of bands: {len(cube.wavelengths)}")

In [ ]:
# Plot spectrum at a sample location
cube.plot_spectrum(
    use_corrected=True,
    wavelength_range=None,
    interpolate_wavelengths=None,
    normalize_method=None,
    legend_loc="outside",
    use_inline_labels=False,
    show_std=True,
    track_index=300,
    slit_index=50,
    ylim=0.2,
)

### 2.4 Apply Spectral Smoothing (Optional)

In [ ]:
# Step 2: Smooth with moving average (window=10)
cube.apply_spectral_smoothing(wavelength_smoothing=10, method="moving_average")

print(f"✅ Spectral smoothing complete")
print(f"   Shape after smoothing: {cube.data_corrected.shape}")

### 2.5 Inspect Spectrum

In [ ]:
# Plot spectrum at a sample location
cube.plot_spectrum(
    use_corrected=True,
    wavelength_range=None,
    interpolate_wavelengths=None,
    normalize_method=None,
    legend_loc="outside",
    use_inline_labels=False,
    show_std=True,
    track_index=300,
    slit_index=50,
    ylim=0.2,
)

### 2.6 Apply L2 Normalization

In [ ]:
# Step 3: Normalize with L2 (unit-length spectra)
cube.apply_spectral_normalization(method="l2")

print(f"✅ L2 normalization complete")
print(
    f"   Value range: [{cube.data_corrected.min():.4f}, {cube.data_corrected.max():.4f}]"
)

### 2.7 Inspect Normalized Spectrum

In [ ]:
# Plot normalized spectrum
cube.plot_spectrum(
    use_corrected=True,
    wavelength_range=None,
    interpolate_wavelengths=None,
    normalize_method=None,
    legend_loc="outside",
    use_inline_labels=False,
    show_std=True,
    track_index=300,
    slit_index=50,
)

## 3. Prepare Data for NDI Analysis

In [ ]:
print("=" * 60)
print("🔬 PREPARING DATA FOR NDI ANALYSIS")
print("=" * 60)

# Get datacube
data = cube.data_corrected
wavelengths = cube.wavelengths

print(f"\nDatacube shape: {data.shape}")
print(f"Wavelength range: {wavelengths[0]:.1f} - {wavelengths[-1]:.1f} nm")
print(f"Number of bands: {len(wavelengths)}")

### 3.1 Check Wavelength Availability

In [ ]:
# Diagnostic: Check if all required wavelengths are available
print("\n🔍 Checking wavelength availability...")
required_wavelengths = {
    "500 nm (Rust)": 500,
    "550 nm (Chl-a, Cyano)": 550,
    "600 nm (Rust)": 600,
    "620 nm (Cyano)": 620,
    "675 nm (Chl-a)": 675,
}

all_available = True
for name, target_wl in required_wavelengths.items():
    closest_idx = np.argmin(np.abs(wavelengths - target_wl))
    closest_wl = wavelengths[closest_idx]
    difference = abs(closest_wl - target_wl)

    if difference > 20:  # More than 20 nm away
        print(
            f"   ⚠️  {name}: Target {target_wl} nm → using {closest_wl:.1f} nm (Δ={difference:.1f} nm) - MAY BE INACCURATE"
        )
        all_available = False
    else:
        print(
            f"   ✅ {name}: Target {target_wl} nm → using {closest_wl:.1f} nm (Δ={difference:.1f} nm)"
        )

if all_available:
    print("\n✅ All required wavelengths are available!")
else:
    print(
        "\n⚠️  Some wavelengths are far from target - NDI results may be less accurate"
    )

### 3.2 Extract Reflectance at Key Wavelengths

In [ ]:
# Extract reflectance at key wavelengths using utils function
print("\n📍 Extracting reflectance at key wavelengths...")

idx_500 = find_closest_wavelength_index(wavelengths, 500)
idx_525 = find_closest_wavelength_index(wavelengths, 525)
idx_550 = find_closest_wavelength_index(wavelengths, 550)
idx_600 = find_closest_wavelength_index(wavelengths, 600)
idx_620 = find_closest_wavelength_index(wavelengths, 620)
idx_675 = find_closest_wavelength_index(wavelengths, 675)

R500 = data[:, :, idx_500]
R525 = data[:, :, idx_525]
R550 = data[:, :, idx_550]
R600 = data[:, :, idx_600]
R620 = data[:, :, idx_620]
R675 = data[:, :, idx_675]

print("\n✅ Reflectance values extracted!")

## 4. Compute NDI Maps (0-1 Normalized)

Using functions from `ndi_analysis_utils.py`

In [ ]:
print("=" * 60)
print("🔬 COMPUTING NDI MAPS (0-1 Normalized)")
print("=" * 60)

### 4.1 Rust NDI

In [ ]:
# 1. RUST NDI: (R600 - R500) / (R600 + R500)
print("\n" + "=" * 60)
print("1️⃣ RUST NDI - Fe oxide corrosion detection")
print("=" * 60)
print("Formula: (R600 - R500) / (R600 + R500)")
print("Detects: Oxidized metal surfaces (rust)")

rust_ndi = compute_ndi(R600, R500, name="Rust NDI")

In [ ]:
# 1. RUST NDI: (R600 - R500) / (R600 + R500)
print("\n" + "=" * 60)
print("1️⃣ RUST NDI - Fe oxide corrosion detection")
print("=" * 60)
print("Formula: (R600 - R500) / (R600 + R500)")
print("Detects: Oxidized metal surfaces (rust)")

rust_ndi_525 = compute_ndi(R675, R525, name="Rust NDI")

In [ ]:
# 1. RUST NDI: (R600 - R500) / (R600 + R500)
print("\n" + "=" * 60)
print("1️⃣ RUST NDI - Fe oxide corrosion detection")
print("=" * 60)
print("Formula: (R600 - R500) / (R600 + R500)")
print("Detects: Oxidized metal surfaces (rust)")

rust_ndi_550 = compute_ndi(R675, R550, name="Rust NDI")

### 4.2 Rust 1st Derivative

In [ ]:
# 2. RUST 1ST DERIVATIVE: dR/dλ between 500-600 nm
print("\n" + "=" * 60)
print("2️⃣ RUST 1ST DERIVATIVE - Corrosion slope")
print("=" * 60)
print("Formula: dR/dλ between 500-600 nm")
print("Detects: Steepness of red slope (more rust = steeper)")

rust_derivative = compute_first_derivative(
    data, wavelengths, 500, 600, name="Rust 1st Derivative"
)

### 4.3 Chlorophyll-a NDI

In [ ]:
# 3. CHLOROPHYLL-A NDI: (R550 - R675) / (R550 + R675)
print("\n" + "=" * 60)
print("3️⃣ CHLOROPHYLL-A NDI - Photosynthetic biofilm")
print("=" * 60)
print("Formula: (R550 - R675) / (R550 + R675)")
print("Detects: Algae/biofilm with chlorophyll-a pigment")

chla_ndi = compute_ndi(R550, R675, name="Chlorophyll-a NDI")

### 4.4 Chlorophyll-a 2nd Derivative

In [ ]:
# 4. CHLOROPHYLL-A 2ND DERIVATIVE: d²R/dλ² at 675 nm
print("\n" + "=" * 60)
print("4️⃣ CHLOROPHYLL-A 2ND DERIVATIVE - Refined Chl-a detection")
print("=" * 60)
print("Formula: d²R/dλ² at 675 nm")
print("Detects: Curvature of Chl-a absorption trough (enhanced contrast)")

chla_second_deriv = compute_second_derivative_at_wavelength(
    data, wavelengths, 675, window=2, name="Chlorophyll-a 2nd Derivative"
)

### 4.5 Cyanobacteria NDI

In [ ]:
# 5. CYANOBACTERIA NDI: (R550 - R620) / (R550 + R620)
print("\n" + "=" * 60)
print("5️⃣ CYANOBACTERIA NDI - Microbial mats")
print("=" * 60)
print("Formula: (R550 - R620) / (R550 + R620)")
print("Detects: Phycobilin pigments (phycoerythrin/phycocyanin) from metal leakage")

cyano_ndi = compute_ndi(R550, R620, name="Cyanobacteria NDI")

### 4.6 Brightness

In [ ]:
# 6. BRIGHTNESS: Mean of 490-690 nm
print("\n" + "=" * 60)
print("6️⃣ BRIGHTNESS - White encrustations")
print("=" * 60)
print("Formula: Mean reflectance across all bands (490-690 nm)")
print("Detects: Carbonate deposits, barnacles, white sediment")

brightness = compute_brightness(data, name="Brightness")

print("\n" + "=" * 60)
print("✅ ALL NDI MAPS COMPUTED!")
print("=" * 60)

## 5. Visualize NDI Maps (Individual Plots)

In [ ]:
# 1. Rust NDI
plot_ndi_heatmap(
    rust_ndi,
    title="Rust NDI - (R600-R500)/(R600+R500)\nDetects: Fe oxide corrosion",
    cmap="Reds",
    figsize=(24, 10),
)

In [ ]:
# 1. Rust NDI
plot_ndi_heatmap(
    rust_ndi_525,
    title="Rust",
    cmap="Reds",
    figsize=(24, 10),
)

In [ ]:
# 1. Rust NDI
plot_ndi_heatmap(
    rust_ndi_550,
    title="Rust",
    cmap="Reds",
    figsize=(24, 10),
)

In [ ]:
# 2. Rust 1st Derivative
plot_ndi_heatmap(
    rust_derivative,
    title="Rust 1st Derivative - dR/dλ (500-600 nm)\nDetects: Corrosion slope",
    cmap="YlOrRd",
    figsize=(24, 10),
)

In [ ]:
# 3. Chlorophyll-a NDI
plot_ndi_heatmap(
    chla_ndi,
    title="Chlorophyll-a NDI - (R550-R675)/(R550+R675)\nDetects: Photosynthetic biofilm",
    cmap="Greens",
    figsize=(24, 10),
)

In [ ]:
# 4. Chlorophyll-a 2nd Derivative
plot_ndi_heatmap(
    chla_second_deriv,
    title="Chlorophyll-a 2nd Derivative - d²R/dλ² at 675 nm\nDetects: Refined Chl-a (trough curvature)",
    cmap="YlGn",
    figsize=(24, 10),
)

In [ ]:
# 5. Cyanobacteria NDI
plot_ndi_heatmap(
    cyano_ndi,
    title="Cyanobacteria NDI - (R550-R620)/(R550+R620)\nDetects: Phycobilin pigments (microbial mats)",
    cmap="PuOr",
    figsize=(24, 10),
)

In [ ]:
# 6. Brightness
plot_ndi_heatmap(
    brightness,
    title="Brightness - Mean(490-690 nm)\nDetects: White encrustations/carbonate",
    cmap="gray",
    figsize=(24, 10),
)

### 5.1 Grid View of All NDI Maps

In [ ]:
# Plot all NDI maps in grid
ndi_maps = [
    rust_ndi,
    rust_derivative,
    chla_ndi,
    chla_second_deriv,
    cyano_ndi,
    brightness,
]

titles = [
    "1. Rust NDI\n(R600-R500)/(R600+R500)",
    "2. Rust 1st Derivative\ndR/dλ (500-600 nm)",
    "3. Chlorophyll-a NDI\n(R550-R675)/(R550+R675)",
    "4. Chl-a 2nd Derivative\nd²R/dλ² at 675 nm",
    "5. Cyanobacteria NDI\n(R550-R620)/(R550+R620)",
    "6. Brightness\nMean(490-690 nm)",
]

plot_all_ndi_grid(ndi_maps, titles, figsize=(28, 24))

## 6. Compute Raw Band Ratios (NOT Normalized)

In [ ]:
print("=" * 60)
print("🔬 COMPUTING RAW BAND RATIOS")
print("=" * 60)

# 1. RUST RATIO: R600 / R500
print("\n" + "=" * 60)
print("1️⃣ RUST RATIO - R600/R500")
print("=" * 60)
rust_ratio = compute_band_ratio(R600, R500, name="Rust Ratio (R600/R500)")

# 2. CHLOROPHYLL-A RATIO: R550 / R675
print("\n" + "=" * 60)
print("2️⃣ CHLOROPHYLL-A RATIO - R550/R675")
print("=" * 60)
chla_ratio = compute_band_ratio(R550, R675, name="Chl-a Ratio (R550/R675)")

# 3. CYANOBACTERIA RATIO: R550 / R620
print("\n" + "=" * 60)
print("3️⃣ CYANOBACTERIA RATIO - R550/R620")
print("=" * 60)
cyano_ratio = compute_band_ratio(R550, R620, name="Cyano Ratio (R550/R620)")

print("\n" + "=" * 60)
print("✅ ALL RAW RATIOS COMPUTED!")
print("=" * 60)

### 6.1 Visualize Raw Band Ratios

In [ ]:
# 1. Rust Ratio
plot_raw_ratio_heatmap(
    rust_ratio,
    title="Rust Ratio - R600/R500 (Raw)\nDetects: Red/green ratio (rust signature)",
    cmap="Reds",
    figsize=(24, 10),
)

In [ ]:
# 2. Chlorophyll-a Ratio
plot_raw_ratio_heatmap(
    chla_ratio,
    title="Chlorophyll-a Ratio - R550/R675 (Raw)\nDetects: Green/red ratio (chlorophyll absorption)",
    cmap="Greens",
    figsize=(24, 10),
)

In [ ]:
# 3. Cyanobacteria Ratio
plot_raw_ratio_heatmap(
    cyano_ratio,
    title="Cyanobacteria Ratio - R550/R620 (Raw)\nDetects: Green/orange ratio (phycobilin absorption)",
    cmap="PuOr",
    figsize=(24, 10),
)

## 7. Compute Baseline-Centered NDI Maps ([-1, +1] with mean=0)

In [ ]:
print("=" * 60)
print("🔬 COMPUTING BASELINE-CENTERED NDI MAPS")
print("=" * 60)
print("Normalization: [-1, +1] with mean=0 (white in plots)")
print("=" * 60)

# 1. RUST NDI (baseline-centered)
print("\n" + "=" * 60)
print("1️⃣ RUST NDI - Baseline-Centered")
print("=" * 60)
rust_ndi_bc = compute_ndi_baseline_centered(R600, R500, name="Rust NDI")

# 2. RUST 1ST DERIVATIVE (baseline-centered)
print("\n" + "=" * 60)
print("2️⃣ RUST 1ST DERIVATIVE - Baseline-Centered")
print("=" * 60)
rust_derivative_bc = compute_first_derivative_baseline_centered(
    data, wavelengths, 500, 600, name="Rust 1st Derivative"
)

# 3. CHLOROPHYLL-A NDI (baseline-centered)
print("\n" + "=" * 60)
print("3️⃣ CHLOROPHYLL-A NDI - Baseline-Centered")
print("=" * 60)
chla_ndi_bc = compute_ndi_baseline_centered(R550, R675, name="Chlorophyll-a NDI")

# 4. CHLOROPHYLL-A 2ND DERIVATIVE (baseline-centered)
print("\n" + "=" * 60)
print("4️⃣ CHLOROPHYLL-A 2ND DERIVATIVE - Baseline-Centered")
print("=" * 60)
chla_second_deriv_bc = compute_second_derivative_baseline_centered(
    data, wavelengths, 675, window=2, name="Chlorophyll-a 2nd Derivative"
)

# 5. CYANOBACTERIA NDI (baseline-centered)
print("\n" + "=" * 60)
print("5️⃣ CYANOBACTERIA NDI - Baseline-Centered")
print("=" * 60)
cyano_ndi_bc = compute_ndi_baseline_centered(R550, R620, name="Cyanobacteria NDI")

# 6. BRIGHTNESS (baseline-centered)
print("\n" + "=" * 60)
print("6️⃣ BRIGHTNESS - Baseline-Centered")
print("=" * 60)
brightness_bc = compute_brightness_baseline_centered(data, name="Brightness")

print("\n" + "=" * 60)
print("✅ ALL BASELINE-CENTERED NDI MAPS COMPUTED!")
print("=" * 60)

### 7.1 Visualize Baseline-Centered NDI Maps

In [ ]:
# 1. Rust NDI (baseline-centered) - Use RdBu_r (Blue → White → Red)
plot_ndi_baseline_centered(
    rust_ndi_bc,
    title="Rust NDI (Baseline-Centered)\n0=normal (white), +1=high rust (red), -1=low (blue)",
    cmap="RdBu_r",  # Blue=low, White=normal, Red=high rust
    figsize=(24, 10),
)

In [ ]:
# 2. Rust 1st Derivative (baseline-centered)
plot_ndi_baseline_centered(
    rust_derivative_bc,
    title="Rust 1st Derivative (Baseline-Centered)\n0=normal slope (white), +1=steep red slope (red)",
    cmap="RdYlGn_r",  # Green=negative slope, White=normal, Red=steep positive slope
    figsize=(24, 10),
)

In [ ]:
# 3. Chlorophyll-a NDI (baseline-centered) - Use PiYG (Pink → White → Green)
plot_ndi_baseline_centered(
    chla_ndi_bc,
    title="Chlorophyll-a NDI (Baseline-Centered)\n0=normal (white), +1=high chlorophyll (green), -1=low (pink)",
    cmap="PiYG",  # Pink=low, White=normal, Green=high chlorophyll
    figsize=(24, 10),
)

In [ ]:
# 4. Chlorophyll-a 2nd Derivative (baseline-centered)
plot_ndi_baseline_centered(
    chla_second_deriv_bc,
    title="Chlorophyll-a 2nd Derivative (Baseline-Centered)\n0=normal curvature (white), +1=strong absorption (green)",
    cmap="PiYG",  # Pink=low, White=normal, Green=strong Chl-a absorption
    figsize=(24, 10),
)

In [ ]:
# 5. Cyanobacteria NDI (baseline-centered) - Use PuOr_r (Orange → White → Purple)
plot_ndi_baseline_centered(
    cyano_ndi_bc,
    title="Cyanobacteria NDI (Baseline-Centered)\n0=normal (white), +1=high cyanobacteria (purple), -1=low (orange)",
    cmap="PuOr_r",  # Orange=low, White=normal, Purple=high cyanobacteria
    figsize=(24, 10),
)

In [ ]:
# 6. Brightness (baseline-centered) - Use "RdGy_r"
plot_ndi_baseline_centered(
    brightness_bc,
    title="Brightness (Baseline-Centered)\n0=normal brightness (white), +1=bright (red), -1=dark (gray)",
    cmap="RdGy_r",  # Gray=low brightness, White=normal, Red=high brightness
    figsize=(24, 10),
)

## 8. Summary

This notebook demonstrated the complete NDI analysis workflow using the refactored `ndi_analysis_utils.py` module:

✅ **Data Loading & Preprocessing:**
- Illumination correction
- Wavelength filtering (490-690 nm)
- Spectral smoothing (moving average, window=10)
- L2 normalization

✅ **NDI Computation (3 versions):**
1. **Standard NDI (0-1 normalized)** - Good for absolute detection
2. **Raw Band Ratios** - Preserves original magnitude
3. **Baseline-Centered NDI ([-1, +1] with mean=0)** - Best for anomaly detection

✅ **Indices Computed:**
- Rust NDI (R600/R500)
- Rust 1st Derivative (500-600 nm slope)
- Chlorophyll-a NDI (R550/R675)
- Chlorophyll-a 2nd Derivative (675 nm curvature)
- Cyanobacteria NDI (R550/R620)
- Brightness (mean reflectance)

All functions are now cleanly organized in `utils/ndi_analysis_utils.py` for easy reuse! 🎉

## 9. Georeferenced NDI Visualization (NEW!)

Now let's use the new **georeferenced plotting functions** that use `plot_georef` under the hood. This gives proper spatial coordinates instead of just track/slit indices.

You can toggle between georeferenced (`use_georef=True`) and simple plotting (`use_georef=False`).

### 9.1 Georeferenced Rust NDI (0-1 Normalized)

In [ ]:
# Plot Rust NDI with georeferencing
fig = plot_ndi_georef(
    cube=cube,
    ndi_map=rust_ndi,
    title="Rust NDI - Georeferenced (R600-R500)/(R600+R500)",
    cmap="Reds",
    vmin=0,
    vmax=1,
    use_georef=True,  # Toggle: True for georeferenced, False for simple plot
    figsize=(24, 10),
    coordinate_system="NED",
    apply_alignment_shift=True,
)

### 9.2 Georeferenced Chlorophyll-a NDI

In [ ]:
# Plot Chlorophyll-a NDI with georeferencing
fig = plot_ndi_georef(
    cube=cube,
    ndi_map=chla_ndi,
    title="Chlorophyll-a NDI - Georeferenced (R550-R675)/(R550+R675)",
    cmap="Greens",
    vmin=0,
    vmax=1,
    use_georef=True,
    figsize=(24, 10),
    coordinate_system="NED",
    apply_alignment_shift=True,
)

### 9.3 Georeferenced Baseline-Centered Rust NDI

In [ ]:
# Plot Baseline-Centered Rust NDI with georeferencing and diverging colormap
fig = plot_ndi_baseline_centered_georef(
    cube=cube,
    ndi_map=rust_ndi_bc,
    title="Rust NDI - Baseline-Centered Georeferenced\n0=normal (white), +1=high rust (red), -1=low (blue)",
    cmap="RdBu_r",
    use_georef=True,
    figsize=(24, 10),
    coordinate_system="NED",
    apply_alignment_shift=True,
)

### 9.4 Georeferenced Baseline-Centered Chlorophyll-a NDI

In [ ]:
# Plot Baseline-Centered Chlorophyll-a NDI with georeferencing
fig = plot_ndi_baseline_centered_georef(
    cube=cube,
    ndi_map=chla_ndi_bc,
    title="Chlorophyll-a NDI - Baseline-Centered Georeferenced\n0=normal (white), +1=high chlorophyll (green), -1=low (pink)",
    cmap="PiYG",
    use_georef=True,
    figsize=(24, 10),
    coordinate_system="NED",
    apply_alignment_shift=True,
)

### 9.5 Toggle Between Georeferenced and Simple Plots

You can easily switch between georeferenced and simple plotting by changing `use_georef`:

In [ ]:
# Simple (non-georeferenced) plot - faster, just track/slit indices
fig = plot_ndi_georef(
    cube=cube,
    ndi_map=brightness,
    title="Brightness - Simple Plot (Track/Slit Indices)",
    cmap="gray",
    vmin=0,
    vmax=1,
    use_georef=False,  # Simple plotting without georeferencing
    figsize=(24, 10),
)

## 10. Summary - Georeferenced vs Simple Plotting

**NEW Georeferenced Functions:**
- `plot_ndi_georef()` - For 0-1 normalized NDI maps with proper coordinates
- `plot_ndi_baseline_centered_georef()` - For baseline-centered NDI maps with diverging colormaps

**Key Parameters:**
- `use_georef=True` → Uses `plot_georef` with proper spatial coordinates (E/N in meters or Lat/Lon)
- `use_georef=False` → Simple `imshow` with track/slit indices (faster)
- `coordinate_system="NED"` → Choose "NED", "ECEF", or "LATLON"
- `apply_alignment_shift=True` → Apply UHI alignment corrections from config

**Advantages of Georeferenced Plotting:**
✅ Proper spatial coordinates (meters or lat/lon)
✅ Can overlay with other georeferenced data (MBES, navigation, etc.)
✅ Matches exactly with RGB `plot_georef()` output
✅ Ready for GIS export and spatial analysis

**When to Use Simple Plotting:**
- Quick visualization during development
- When you only care about relative positions
- Faster rendering for large datasets

All functions are now in `utils/ndi_analysis_utils.py` for easy reuse! 🎉